# FLUX Phase 2: Resonance Field Core (RFC)
## Complete Pipeline — Train, Test, Demo, Upload

> **Goal:** Replace all weight matrices with a dynamic 3D field tensor that stores knowledge as energy minima (attractors). Updates are local, no backprop through the field, zero catastrophic forgetting by physics.

### What this notebook does:
1. **Setup** — Clone/pull repo, install deps, detect hardware
2. **Load Phase 1** — Verify CSE checkpoint loads and works
3. **Smoke Test** — Verify ResonanceField builds, perturb/query work
4. **Train** — Projection warmup + field formation (50 patterns × 20 reps)
5. **Upload** — Checkpoint → HuggingFace Hub
6. **Test 1** — Attractor formation + no-forgetting (similarity > 0.7)
7. **Test 2** — Local update verification (far changes < 0.001)
8. **Test 3** — Field stability over 1000 steps (no explosion/collapse)
9. **Demo 1** — Attractor formation visualization
10. **Demo 2** — The no-forgetting demo
11. **Results** — Generate RESULTS_PHASE_2.md
12. **Final** — Upload logs → HuggingFace & push to GitHub

### Secrets Required:
- **`HF_TOKEN`** — HuggingFace write token (Add via Kaggle → Add-ons → Secrets)

---
## Cell 1: Clone / Pull Repository

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = Path("/kaggle/working/FLUX")

if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("ℹ Repo already exists — pulling latest changes...")
    result = subprocess.run(
        ['git', 'pull', '--ff-only'],
        cwd=str(WORK_DIR), capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    print("✓ Pulled latest")
else:
    print(f"ℹ Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("✓ Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")
print(f"Files: {sorted(os.listdir('.'))}")

---
## Cell 2: Install Dependencies & Setup

In [ ]:
!pip install -q datasets rich faiss-cpu huggingface_hub
!python setup.py

---
## Cell 3: Hardware, Secrets & PhaseLogger

In [ ]:
import sys, torch, platform
from pathlib import Path

# Add project paths
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase1'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase2'))

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    load_checkpoint, save_checkpoint, verify_checkpoint_chain,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
    PhaseResults,
)

# Init logger
log = PhaseLogger(phase=2)
log.separator("Phase 2: Resonance Field Core")

# Hardware
device = get_device()
hw = get_hardware_info()
log.cell_start("Cell 3 — Hardware & Secrets")
for k, v in hw.items():
    log.info(f"{k}: {v}")
    print(f"  {k}: {v}")

# HF Token
token = _resolve_hf_token()
if token:
    log.success("HuggingFace token loaded")
    print("  ✓ HF token loaded")
else:
    log.warning("No HuggingFace token found")
    print("  ⚠ No HF token — uploads will be skipped")

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

---
## Cell 4: Load Phase 1 Checkpoint & Smoke Test

In [ ]:
log.cell_start("Cell 4 — Load Phase 1 & Smoke Test")

# Verify Phase 1 checkpoint
assert verify_checkpoint_chain(up_to_phase=2), "Phase 1 checkpoint missing!"

# Load CSE
from cse import ContinuousSemanticEncoder
checkpoint_p1 = load_checkpoint(1)
cse = ContinuousSemanticEncoder(**checkpoint_p1.get('config', {}))
cse.load_state_dict(checkpoint_p1['state_dict'])
cse = cse.to(device).eval()
for p in cse.parameters():
    p.requires_grad = False
log.success(f"CSE loaded: {sum(p.numel() for p in cse.parameters()):,} params")

# Smoke test CSE
wave = cse.encode("The resonance field awakens")
assert wave.full.shape[-1] == 432, f"Bad wave dim: {wave.full.shape}"
assert not torch.isnan(wave.full).any(), "NaN in wave!"
log.success(f"CSE smoke test passed: wave shape {wave.full.shape}")

# Smoke test ResonanceField
from field import ResonanceField, FIELD_H, FIELD_W, FIELD_D, FIELD_FEATURES
field_test = ResonanceField(h=8, w=8, d=8, features=64).to(device)
vec = wave.full.mean(dim=0)
loc = field_test.perturb(vec)
feats, sims, locs = field_test.query(vec)
assert feats.shape[1] == 64
log.success(f"ResonanceField smoke: perturb→({loc.h},{loc.w},{loc.d}), query→{feats.shape}")
del field_test

print("  ✓ Phase 1 CSE loaded and verified")
print("  ✓ ResonanceField builds and runs correctly")
log.cell_end("Cell 4 — Load Phase 1 & Smoke Test", "PASS")

---
## Cell 5: Training — Projection Warmup + Field Formation

This is the main training cell. Three phases:
- **Phase A:** Brief gradient-based warmup of projection layers (~500 steps)
- **Phase B:** Physics-based field formation via repeated perturbation + settling
- **Phase C:** Validation — query field for known patterns, measure retrieval accuracy

In [ ]:
log.cell_start("Cell 5 — Training")
import time
from tqdm import tqdm
import torch.nn.functional as F
from field import ResonanceField, FIELD_H, FIELD_W, FIELD_D, FIELD_FEATURES
from attractor import AttractorCatalog
from field_ops import compute_field_statistics, find_top_attractors, normalize_field_energy

start_time = time.time()

# ── Create ResonanceField ──
field = ResonanceField(
    h=FIELD_H, w=FIELD_W, d=FIELD_D,
    features=FIELD_FEATURES,
).to(device)
total_params = sum(p.numel() for p in field.parameters())
buffer_size = sum(b.numel() for b in field.buffers())
log.info(f"Field: {total_params:,} params, {buffer_size:,} buffers (~{buffer_size*4/1e6:.0f} MB)")
print(f"  Field: {FIELD_H}×{FIELD_W}×{FIELD_D}×{FIELD_FEATURES}")
print(f"  Trainable params: {total_params:,}")
print(f"  Buffer memory: ~{buffer_size*4/1e6:.0f} MB")

# ── Prepare data ──
try:
    from datasets import load_dataset
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    texts = [item['text'].strip()[:200] for item in dataset
             if len(item['text'].strip()) > 50 and not item['text'].strip().startswith('=')][:200]
    log.success(f"Loaded {len(texts)} texts from WikiText-2")
except Exception as e:
    log.warning(f"WikiText-2 unavailable ({e}), using built-in texts")
    texts = [
        "The cat sat on the mat and watched the birds fly past the window.",
        "Quantum mechanics describes the behavior of particles at atomic scales.",
        "The economy grew significantly during the second quarter of the fiscal year.",
        "Machine learning algorithms process data to discover hidden patterns.",
        "The ancient civilization built pyramids that still stand thousands of years later.",
    ] * 40

# ── Encode all texts with CSE ──
print("  Encoding texts with CSE...")
wave_vectors = []
with torch.no_grad():
    for text in tqdm(texts[:100], desc="  Encoding"):
        wave = cse.encode(text)
        vec = wave.full.mean(dim=0)
        wave_vectors.append(vec)
wave_batch = torch.stack(wave_vectors).to(device)
log.success(f"Encoded {wave_batch.shape[0]} texts → {wave_batch.shape}")

# ── Phase A: Projection Warmup ──
print("\n  Phase A: Projection Warmup (500 steps)...")
optimizer = torch.optim.Adam(
    list(field.wave_to_location.parameters()) + list(field.wave_to_feature.parameters()),
    lr=1e-3,
)
for step in range(500):
    idx = torch.randint(0, wave_batch.shape[0], (32,), device=device)
    batch = wave_batch[idx]
    features = field.wave_to_feature(batch)
    wave_sim = F.cosine_similarity(batch.unsqueeze(1), batch.unsqueeze(0), dim=-1)
    feat_sim = F.cosine_similarity(features.unsqueeze(1), features.unsqueeze(0), dim=-1)
    loss_sim = F.mse_loss(feat_sim, wave_sim)
    locations = torch.sigmoid(field.wave_to_location(batch))
    loss_div = -torch.cdist(locations, locations).mean()
    loss = loss_sim + 0.1 * loss_div
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        log.info(f"Warmup {step}/500: loss_sim={loss_sim.item():.4f} loss_div={loss_div.item():.4f}")
        print(f"    Step {step}: loss_sim={loss_sim.item():.4f}, loss_div={loss_div.item():.4f}")
log.success("Projection warmup complete")

# ── Phase B: Field Formation ──
NUM_PATTERNS = 50
REPETITIONS = 20
patterns = wave_batch[:NUM_PATTERNS]
print(f"\n  Phase B: Field Formation ({NUM_PATTERNS} patterns × {REPETITIONS} reps)...")
with torch.no_grad():
    total_steps = 0
    for rep in range(REPETITIONS):
        for i in range(NUM_PATTERNS):
            field.perturb(patterns[i])
            total_steps += 1
        if total_steps % 50 == 0:
            field.settle(steps=10)
        if rep % 5 == 0 or rep == REPETITIONS - 1:
            stats = field.get_field_stats()
            msg = f"Rep {rep+1}/{REPETITIONS}: attractors={stats['num_attractors']}, energy={stats['total_energy']:.1f}, max_mass={stats['max_mass']:.4f}"
            log.info(msg)
            print(f"    {msg}")
    field.settle(steps=50)
    normalize_field_energy(field)
log.success(f"Field formation complete: {total_steps} perturbations")

# ── Phase C: Validation ──
print("\n  Phase C: Validation...")
hits = 0
total_sim = 0.0
with torch.no_grad():
    for i in range(NUM_PATTERNS):
        feats, sims, locs = field.query(patterns[i])
        top_sim = sims[0].item() if sims.numel() > 0 else 0.0
        total_sim += top_sim
        if top_sim > 0.3:
            hits += 1
accuracy = hits / NUM_PATTERNS
avg_sim = total_sim / NUM_PATTERNS

# Attractor catalog
catalog = AttractorCatalog(field)
num_discovered = catalog.scan_and_update(mass_threshold=0.1)

elapsed = time.time() - start_time
log.metric("retrieval_accuracy", f"{accuracy:.4f}")
log.metric("avg_similarity", f"{avg_sim:.4f}")
log.metric("num_attractors", field.num_attractors())
log.metric("training_time", f"{elapsed:.1f}s")
print(f"\n  ✓ Training complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"  Retrieval accuracy: {hits}/{NUM_PATTERNS} = {accuracy:.1%}")
print(f"  Average similarity: {avg_sim:.4f}")
print(f"  Attractors: {field.num_attractors()}")
print(f"  Discovered via scan: {num_discovered}")
log.cell_end("Cell 5 — Training", "PASS")

---
## Cell 6: Save Checkpoint & Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 6 — Save & Upload Checkpoint")

field_stats = compute_field_statistics(field)
checkpoint_state = {
    'config': {
        'field': {'h': FIELD_H, 'w': FIELD_W, 'd': FIELD_D, 'features': FIELD_FEATURES, 'wave_dim': 432},
        'cse': checkpoint_p1.get('config', {}),
    },
    'state_dict': field.state_dict(),
    'cse_state_dict': cse.state_dict(),
    'metrics': {
        'retrieval_accuracy': accuracy,
        'avg_similarity': avg_sim,
        'num_attractors': field.num_attractors(),
        'total_energy': field.total_energy(),
        'training_time_seconds': elapsed,
        'field_stats': field_stats,
    },
    'attractor_catalog': catalog.to_dict(),
    'training_config': {
        'num_patterns': NUM_PATTERNS, 'repetitions': REPETITIONS,
        'warmup_steps': 500, 'warmup_lr': 1e-3,
    },
}
save_checkpoint(2, checkpoint_state)

# Also save .flx format
from flux_format import save_flux
save_flux(field, cse, 'checkpoints/phase2.flx', catalog,
          metadata={'steps': field.step_count})

# Upload to HuggingFace Hub
upload_checkpoint_to_hf(phase=2, hf_token=token)

log.cell_end("Cell 6 — Save & Upload Checkpoint", "PASS")

---
## Cells 7–9: Tests

In [ ]:
log.cell_start("Cell 7 — Test 1: Attractor Formation & No-Forgetting")
%run phases/phase2/test_phase2_test1.py
log.cell_end("Cell 7 — Test 1", "PASS")

In [ ]:
log.cell_start("Cell 8 — Test 2: Local Update Verification")
%run phases/phase2/test_phase2_test2.py
log.cell_end("Cell 8 — Test 2", "PASS")

In [ ]:
log.cell_start("Cell 9 — Test 3: Field Stability")
%run phases/phase2/test_phase2_test3.py
log.cell_end("Cell 9 — Test 3", "PASS")

---
## Cells 10–11: Demos

In [ ]:
log.cell_start("Cell 10 — Demo 1: Attractor Formation Visualization")
%run phases/phase2/demo_phase2_demo1.py
from IPython.display import Image, display
display(Image('phases/phase2/demo1_attractor_formation.png'))
log.cell_end("Cell 10 — Demo 1", "PASS")

In [ ]:
log.cell_start("Cell 11 — Demo 2: No-Forgetting Demo")
%run phases/phase2/demo_phase2_demo2.py
from IPython.display import Image, display
display(Image('phases/phase2/demo2_no_forgetting.png'))
log.cell_end("Cell 11 — Demo 2", "PASS")

---
## Cell 12: Interactive Field Exploration

In [ ]:
log.cell_start("Cell 12 — Interactive Exploration")

# Explore the trained field
print("  Field Statistics:")
stats = compute_field_statistics(field)
for k, v in stats.items():
    print(f"    {k}: {v}")

# Top attractors
print("\n  Top 10 Attractors:")
top = find_top_attractors(field, k=10)
for i, a in enumerate(top):
    print(f"    #{i+1}: loc={a['location']} mass={a['mass']:.4f} energy={a['energy']:.4f}")

# Query with custom text
test_texts = ["The cat sat on the mat", "Quantum physics experiment", "Stock market analysis"]
print("\n  Custom queries:")
for text in test_texts:
    wave = cse.encode(text)
    vec = wave.full.mean(dim=0)
    feats, sims, locs = field.query(vec, k=3)
    loc = field.wave_to_field_coords(vec)
    print(f"    '{text[:30]}...' → ({loc.h},{loc.w},{loc.d}) top_sim={sims[0].item():.4f}")

log.cell_end("Cell 12 — Interactive Exploration", "PASS")

---
## Cell 13: Generate Results & View Log

In [ ]:
log.cell_start("Cell 13 — Results & Log")

# Results will be auto-generated by test scripts via PhaseResults
# Show results file if it exists
results_path = Path('phases/phase2/RESULTS_PHASE_2.md')
if results_path.exists():
    print(results_path.read_text())
else:
    print("  ⚠ Results file not yet generated — run tests first")

# Show full log
print("\n" + "="*60)
print("FULL PHASE 2 LOG:")
print("="*60)
print(log.get_log_contents())

log.cell_end("Cell 13 — Results & Log", "PASS")

---
## Cell 14: Final Upload — Logs to HF Hub & GitHub

In [ ]:
log.cell_start("Cell 14 — Final Upload")

# Upload logs to HF Hub
upload_logs_to_hf(phase=2, hf_token=token)

# Push to GitHub
git_commit_and_push(
    files=['logs/phase2.log', 'phases/phase2/RESULTS_PHASE_2.md'],
    message='Phase 2: Resonance Field Core — training complete'
)

log.cell_end("Cell 14 — Final Upload", "PASS")

---
## Cell 15: Save Artifacts to Kaggle Output

In [ ]:
log.cell_start("Cell 15 — Save Kaggle Artifacts")
import shutil

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

# Copy checkpoint
for f in ['checkpoints/phase2.phase.pt', 'checkpoints/phase2.flx']:
    src = Path(f)
    if src.exists():
        shutil.copy2(str(src), str(output_dir / src.name))
        print(f"  ✓ Copied {src.name} ({src.stat().st_size/1e6:.1f} MB)")

# Copy results and logs
for f in ['phases/phase2/RESULTS_PHASE_2.md', 'logs/phase2.log']:
    src = Path(f)
    if src.exists():
        shutil.copy2(str(src), str(output_dir / src.name))
        print(f"  ✓ Copied {src.name}")

# Copy demo images
for f in Path('phases/phase2').glob('*.png'):
    shutil.copy2(str(f), str(output_dir / f.name))
    print(f"  ✓ Copied {f.name}")

print(f"\n  Kaggle output: {list(output_dir.iterdir())}")
log.cell_end("Cell 15 — Save Kaggle Artifacts", "PASS")

print("\n" + "="*60)
print("  PHASE 2 COMPLETE ✓")
print("="*60)